In [2]:
import sys
sys.path.append('../')

import scqubits as scq
import pandas as pd
import qutip as qt
import numpy as np
from matplotlib import pyplot as plt
from qutip.qip.operations import rz, cz_gate
import cmath
from tqdm import tqdm
from matplotlib.colors import LogNorm
import datetime
import pytz
import scqubits.settings as settings
settings.OVERLAP_THRESHOLD = 0.3
from joblib import Parallel, delayed
import itertools
import scipy.sparse as ssp
from sympy import symbols
import scipy as sp
import utils_2Q_gate_zp as ut
import os
from datetime import datetime
from multiprocessing import Pool

In [4]:
def process_rows(arr, row_indices):
    return arr[row_indices, :]

a = np.arange(100).reshape(25, 4)
indices = list(range(1, a.shape[0], 4))  # [1, 5, 9, ...]
result = process_rows(a, indices)
result

array([[ 4,  5,  6,  7],
       [20, 21, 22, 23],
       [36, 37, 38, 39],
       [52, 53, 54, 55],
       [68, 69, 70, 71],
       [84, 85, 86, 87]])

In [6]:
a[1::4,]

array([[ 4,  5,  6,  7],
       [20, 21, 22, 23],
       [36, 37, 38, 39],
       [52, 53, 54, 55],
       [68, 69, 70, 71],
       [84, 85, 86, 87]])

In [ ]:
tg = [1,5]
a[tg,:]

array([[ 4,  5,  6,  7],
       [20, 21, 22, 23]])

: 

In [5]:
a

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15],
       [16, 17, 18, 19],
       [20, 21, 22, 23],
       [24, 25, 26, 27],
       [28, 29, 30, 31],
       [32, 33, 34, 35],
       [36, 37, 38, 39],
       [40, 41, 42, 43],
       [44, 45, 46, 47],
       [48, 49, 50, 51],
       [52, 53, 54, 55],
       [56, 57, 58, 59],
       [60, 61, 62, 63],
       [64, 65, 66, 67],
       [68, 69, 70, 71],
       [72, 73, 74, 75],
       [76, 77, 78, 79],
       [80, 81, 82, 83],
       [84, 85, 86, 87],
       [88, 89, 90, 91],
       [92, 93, 94, 95],
       [96, 97, 98, 99]])

In [6]:
f_xgate

,tg,drive_amp_1,drive_amp_2,detune_1,detune_2,f_160_old,f_500_old,f_noise_9_2us,f_noise_100_5us,f_noise_100_50us,...,f_1000,f_truc140_ideal,f_truc140_170us,f_truc140_30us,f_truc140_3us,f_500,f_400,f_300,f_160,f_160_low
0,20.000087,0.249753,0.216297,0.438830,0.490612,-0.605455,-0.605542,-0.597582,-0.604252,-0.608336,...,-0.605732,-0.600911,-0.603384,-0.602384,-0.591712,-0.605527,-0.608099,-0.605448,-0.605442,-0.501350
1,30.002512,0.222626,0.201971,0.389445,0.425332,-0.752172,-0.765258,-0.733223,-0.748356,-0.757815,...,-0.764838,-0.771030,-0.776179,-0.773874,-0.750002,-0.765235,-0.766765,-0.752227,-0.752162,-0.540435
2,40.006173,0.223179,0.160035,0.358809,0.386917,-0.885315,-0.884749,-0.842739,-0.870794,-0.888823,...,-0.884612,-0.895347,-0.900302,-0.896187,-0.854423,-0.884744,-0.888147,-0.885447,-0.885314,-0.948272
3,50.000191,0.215248,0.148153,0.354356,0.376483,-1.122802,-1.133417,-1.019964,-1.078394,-1.118451,...,-1.133495,-1.137367,-1.136974,-1.127507,-1.037398,-1.133435,-1.134196,-1.122909,-1.122802,-1.119765
4,60.000967,0.213711,0.144190,0.358967,0.378127,-1.486890,-1.492045,-1.232360,-1.365494,-1.473539,...,-1.485736,-1.466336,-1.463029,-1.437234,-1.229683,-1.492034,-1.483324,-1.487122,-1.486899,-1.439269
5,70.001044,0.209788,0.131851,0.353189,0.369970,-1.638109,-1.637995,-1.275926,-1.456313,-1.622370,...,-1.640146,-1.640558,-1.636456,-1.592602,-1.288350,-1.638037,-1.640861,-1.638609,-1.638138,-1.469358
6,80.007155,0.210573,0.122946,0.355789,0.371130,-1.699696,-1.667877,-1.265451,-1.472646,-1.679502,...,-1.664384,-1.663285,-1.656901,-1.604150,-1.261164,-1.667895,-1.674808,-1.700173,-1.699658,-1.459169
7,90.009744,0.219753,0.113433,0.351332,0.376525,-1.740706,-1.670380,-1.234715,-1.464243,-1.706693,...,-1.671376,-1.654917,-1.649779,-1.593406,-1.236066,-1.670404,-1.690152,-1.741124,-1.740704,-1.460761
8,100.001683,0.212233,0.113924,0.340021,0.362580,-1.845085,-1.861964,-1.240336,-1.498361,-1.805477,...,-1.863662,-1.893940,-1.886920,-1.787068,-1.288739,-1.862016,-1.885734,-1.845797,-1.844986,-2.069365
9,110.009928,0.211658,0.106989,0.341806,0.362874,-1.904947,-1.860689,-1.223821,-1.501067,-1.857262,...,-1.864547,-1.905513,-1.903472,-1.789436,-1.258964,-1.860662,-1.877049,-1.905897,-1.904842,-2.030143


In [ ]:
tg_list = [1, 5, 9, 13, 17 ]


# drive_phi, drive_theta, truc = False, True, 150
drive_phi, drive_theta, truc = True, False, 150
drive_0 = True

t1_other = 170 # μs
gamma_decay_other =  1 / 1e3 / t1_other
gamma_dephase_other = 1 / 1e3 / t1_other

############################################################
folder = 'data_xgate_theta_3ncut.txt' if drive_theta else 'data_xgate_phi_3ncut.txt'
f_xgate = pd.read_csv('data/'+folder)
params = f_xgate[['tg', 'drive_amp_1', 'drive_amp_2', 'detune_1', 'detune_2'
                    ]].to_numpy()[[tg_list], :]
params

array([[[3.00025120e+01, 2.22626000e-01, 2.01971000e-01, 3.89445000e-01,
         4.25332000e-01],
        [7.00010440e+01, 2.09788000e-01, 1.31851000e-01, 3.53189000e-01,
         3.69970000e-01],
        [1.10009928e+02, 2.11658000e-01, 1.06989000e-01, 3.41806000e-01,
         3.62874000e-01],
        [1.50002399e+02, 2.05268000e-01, 9.45520000e-02, 3.39607000e-01,
         3.55509000e-01],
        [1.90009275e+02, 2.09698000e-01, 8.34950000e-02, 3.35382000e-01,
         3.59146000e-01]]])

: 

In [ ]:
# drive_phi, drive_theta, truc = False, True, 150
drive_phi, drive_theta, truc = True, False, 150
drive_0 = True

t1_other = 170 # μs
gamma_decay_other =  1 / 1e3 / t1_other
gamma_dephase_other = 1 / 1e3 / t1_other

############################################################
folder = 'data_xgate_theta_3ncut.txt' if drive_theta else 'data_xgate_phi_3ncut.txt'
f_xgate = pd.read_csv('data/'+folder)
params = f_xgate[['tg', 'drive_amp_1', 'drive_amp_2', 'detune_1', 'detune_2'
                    ]].to_numpy()[1::4, :]
params

array([[3.00025120e+01, 2.22626000e-01, 2.01971000e-01, 3.89445000e-01,
        4.25332000e-01],
       [7.00010440e+01, 2.09788000e-01, 1.31851000e-01, 3.53189000e-01,
        3.69970000e-01],
       [1.10009928e+02, 2.11658000e-01, 1.06989000e-01, 3.41806000e-01,
        3.62874000e-01],
       [1.50002399e+02, 2.05268000e-01, 9.45520000e-02, 3.39607000e-01,
        3.55509000e-01],
       [1.90009275e+02, 2.09698000e-01, 8.34950000e-02, 3.35382000e-01,
        3.59146000e-01]])

In [22]:
# drive_phi, drive_theta, truc = False, True, 150
drive_phi, drive_theta, truc = True, False, 150
drive_0 = True

t1_other = 170 # μs
gamma_decay_other =  1 / 1e3 / t1_other
gamma_dephase_other = 1 / 1e3 / t1_other

############################################################
folder = 'data_xgate_theta_3ncut.txt' if drive_theta else 'data_xgate_phi_3ncut.txt'
f_xgate = pd.read_csv('data/'+folder)
params = f_xgate[['tg', 'drive_amp_1', 'drive_amp_2', 'detune_1', 'detune_2'
                    ]].to_numpy()[[0], :]

num_cpus, n_job = 4, 1*len(params)
logi_state = [0, 2]
folder = '../../data/3ncut_one_zeropi/'
if drive_0:
    evals = 2*np.pi* scq.read(folder + f'zeropi_0_specdata_truc=1000_3ncut.h5').energy_table
    n_theta = 2*np.pi* scq.read(folder + f'zeropi_0_n_theta_truc=1000_3ncut.h5').matrixelem_table
    n_phi = 2*np.pi* scq.read(folder + f'zeropi_0_n_phi_truc=1000_3ncut.h5').matrixelem_table
else:
    evals = 2*np.pi* scq.read(folder + f'zeropi_1_specdata_truc=1000_3ncut.h5').energy_table
    n_theta = 2*np.pi* scq.read(folder + f'zeropi_1_n_theta_truc=1000_3ncut.h5').matrixelem_table
    n_phi = 2*np.pi* scq.read(folder + f'zeropi_1_n_phi_truc=1000_3ncut.h5').matrixelem_table
evals = evals - evals[0]
H0 = qt.Qobj(np.diag(evals))
if drive_phi:
    w_trans_1 = evals[9] - evals[0]
    w_trans_2 = evals[9] - evals[2]
    drive_term = n_phi
if drive_theta:
    w_trans_1 = evals[7] - evals[0]
    w_trans_2 = evals[7] - evals[2]
    drive_term = n_theta

############################################################
thresh = 0.01
hspace_charge = [0, 2]  # Start with the ground and first excited states
for s in hspace_charge:
    for i in range(truc):
        if np.abs(drive_term[s, i] / (2 * np.pi)) > thresh and i not in hspace_charge:
            hspace_charge.append(i)
hspace_charge.sort()
############################################################
# hspace_charge = np.arange(truc).tolist()

############################################################
hspace_len = len(hspace_charge)
logi_idx = [hspace_charge.index(s) for s in logi_state]
H0_truc = ut.truncate_2(H0, hspace_charge)
drive_truc = ut.truncate_2(drive_term, hspace_charge)
H_qbt_drive = [H0_truc, [drive_truc, ut.drive_gauss_A],
                        [drive_truc, ut.drive_gauss_B],]
############################################################
print('params =')
for para in params:
    print(para.tolist(), ',')
print('hspace_len=', hspace_len)
print(' hspace_charge = [')
for i in range(0, len(hspace_charge), 10):
    print(', '.join(map(str, hspace_charge[i:i+10])), ',')
print(']')

params =
[20.000087, 0.249753, 0.216297, 0.43883, 0.490612] ,
hspace_len= 81
 hspace_charge = [
0, 2, 3, 4, 8, 9, 10, 11, 15, 16 ,
18, 19, 20, 23, 24, 25, 28, 31, 33, 35 ,
37, 39, 40, 42, 43, 46, 47, 49, 51, 53 ,
55, 56, 57, 60, 62, 64, 66, 67, 69, 70 ,
73, 76, 77, 79, 81, 83, 85, 86, 88, 91 ,
92, 95, 96, 98, 100, 101, 102, 104, 106, 108 ,
110, 112, 113, 116, 118, 120, 121, 123, 126, 128 ,
130, 132, 133, 136, 137, 139, 141, 143, 145, 146 ,
148 ,
]


### New model - decay to not just $\ket{0}$

In [23]:
folder_1 = 'data/data_gamma_'
folder_2 = 'theta.txt' if drive_theta else 'phi_truc200.txt'
gamma_new = pd.read_csv(folder_1 + folder_2)
gamma_dephase_new = gamma_new['tphi_50us_02'].to_numpy()

if drive_theta:
    Gamma = gamma_decay_other / (np.abs(n_theta[4,7])**2)
else:
    Gamma = gamma_decay_other / (np.abs(n_phi[4,9])**2)
gamma_decay_new = Gamma* np.abs(drive_truc.full())**2    
gamma_dephase_new = gamma_dephase_new *50 /t1_other
jump_t1   = []
jump_tphi = []
for i in range(1,hspace_len):
    for j in range(0,i):
        jump_t1.append( np.sqrt(gamma_decay_new[i,j]) * qt.basis(hspace_len,j) * qt.basis(hspace_len,i).dag() )
    jump_tphi.append( np.sqrt(2*gamma_dephase_new[i]) * qt.basis(hspace_len,i).proj() )
print('np.shape(jump_t1)=',  np.shape(jump_t1), '; np.shape(jump_tphi)=',  np.shape(jump_tphi)) 

np.shape(jump_t1)= (3240, 81, 81) ; np.shape(jump_tphi)= (80, 81, 81)


### noise simulation

In [24]:
c_op_list = [qt.Qobj(np.zeros((truc, truc)))]
c_op_list = []
args = [H_qbt_drive, w_trans_1, w_trans_2, num_cpus, c_op_list, logi_idx]
f_ideal = Parallel(n_jobs=n_job)(delayed(ut.xgate_fidelity_log_noise)(args_indep, *args)
                                            for args_indep in params)
print('\nf_ideal = [')
for i in range(0, len(f_ideal), 4):
    print(', '.join(map(str, f_ideal[i:i+4])), ',')
print(']')
print("Current Mountain Time:", datetime.now(pytz.timezone('America/Denver')))

############################################################
c_op_list = jump_t1 + jump_tphi
args = [H_qbt_drive, w_trans_1, w_trans_2, num_cpus, c_op_list, logi_idx]
f_noise = Parallel(n_jobs=n_job)(delayed(ut.xgate_fidelity_log_noise)(args_indep, *args)
                                            for args_indep in params)
print('\nf_noise = [')
for i in range(0, len(f_noise), 4):
    print(', '.join(map(str, f_noise[i:i+4])), ',')
print(']')
print("Current Mountain Time:", datetime.now(pytz.timezone('America/Denver')))


f_ideal = [
-0.483856699188308 ,
]
Current Mountain Time: 2025-06-13 12:38:04.109427-06:00

f_noise = [
-0.4836847114333318 ,
]
Current Mountain Time: 2025-06-13 12:39:27.444467-06:00


In [25]:
# print(np.shape(n_theta), np.shape(n_phi), np.shape(evals))


In [ ]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)

: 

### Old model - decay to $\ket{0}$

In [ ]:
# ### old model
# idx_2 = 2 if drive_theta else 1

# hspace_len = len(hspace_charge)
# if drive_theta:
#     gamma_decay_old   = [0, gamma_decay_other,  gamma_decay_logi]  + [gamma_decay_other]  * (hspace_len-3)
#     gamma_dephase_old = [0, gamma_dephase_other, gamma_dephase_logi] + [gamma_dephase_other] * (hspace_len-3)
# else:
#     gamma_decay_old   = [0,  gamma_decay_logi]  + [gamma_decay_other]  * (hspace_len-2)
#     gamma_dephase_old = [0,  gamma_dephase_logi] + [gamma_dephase_other] * (hspace_len-2)

# folder_1 = 'data/data_gamma_'
# folder_2 = 'theta.txt' if drive_theta else 'phi.txt'
# gamma_new = pd.read_csv(folder_1 + folder_2)

# ### 't1_50us_47', 't1_50us_27', 't1_50us_07'
# ### 'tphi_50us_02', 'tphi_50us_07', 'tphi_1e6'
# gamma_decay_new = gamma_new['t1_50us_47'].to_numpy()
# # gamma_dephase_new = gamma_new['tphi_1e6'].to_numpy()
# gamma_dephase_new = gamma_new['tphi_50us_02'].to_numpy()

# print("gamma_decay_new[2] = ", gamma_decay_new[2], ", gamma_dephase_new[2] = ", gamma_dephase_new[2])
# gamma_decay_new = gamma_decay_new *50 /t1_other
# gamma_dephase_new = gamma_dephase_new *50 /t1_other

# jump_t1   = []
# jump_tphi = []
# for i in range(1,hspace_len):
#     jump_t1.append( np.sqrt(gamma_decay_new[i]) * qt.basis(hspace_len,0) * qt.basis(hspace_len,i).dag() )
#     jump_tphi.append( np.sqrt(2*gamma_dephase_new[i]) * qt.basis(hspace_len,i).proj() )


In [ ]:
# np.savez("data/data_xgate_theta_collapse_op.npz", arr1=c_op_list)